<a href="https://colab.research.google.com/github/swapnasaina274/malicious-mobile-app-detection/blob/main/02_Advanced_Deep_Learning_and_Risk_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Deep Learning & Security Risk Analysis
## Malicious Mobile Application Detection — CICMalDroid 2020

This notebook extends the existing baseline with leakage-safe preprocessing, advanced ML, deep learning, hybrid ensemble prediction, SHAP explainability, security-risk regression, uncertainty estimation, and an interface for the later LangChain agent layer.

**Important:** the 0–100 risk score below is an engineered operational score, not a ground-truth label supplied by CICMalDroid. For a publishable risk-regression experiment, replace it with independently annotated analyst risk labels.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', 'November-EOD.xlsx', 'TimeTable 2nd sem.xlsx', 'ASIT-250320100080.MOV', 'Asit_Kumar_Panda_CV.pdf', 'Adobe Scan 3 May 2026.pdf', 'MCA 2025 Batch 2nd Sem.xls', 'Asit panda .pages', '250320100080MCA.jpg', 'Baseline Evaluation Test Result - 6  __ CUTM x GfG.xlsx', 'Asit_Kumar_Panda_Resume.docx', 'Final-06-sept-2026.xlsx', 'Dataset']


In [8]:
import os

print(os.listdir('/content/drive/MyDrive/Dataset'))

['feature_vectors_syscalls_frequency_5_Cat.csv', 'feature_vectors_syscallsbinders_frequency_5_Cat.csv', 'syscall_unique.csv', 'feature_vectors_static.csv']


In [2]:
# 0. Setup
import os, json, random, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
ARTIFACT_DIR = Path("artifacts"); ARTIFACT_DIR.mkdir(exist_ok=True)

# If needed in Colab:
# !pip -q install xgboost lightgbm catboost shap
print("Ready. Seed:", SEED)

Ready. Seed: 42


In [4]:
# 1. Dataset path
DATA_PATH = "/content/drive/MyDrive/Dataset/feature_vectors_static.csv"

# 2. Read only 5 rows
import pandas as pd

df_test = pd.read_csv(DATA_PATH, nrows=5)

print("Shape of sample:", df_test.shape)
print("\nColumns:")
print(df_test.columns.tolist())

display(df_test.head())

Shape of sample: (5, 50621)

Columns:
['Unnamed: 0', "' a:targetActivity+AD0-'com.mplus.lib.ui.main.MainActivity' tools:targetApi+AD0-'25'+AD4AXA-n            +ADw-intent+AC0-filter+AD4AXA-n                +ADw-action a:name+AD0-'android.intent.action.MAIN'/+AD4AXA-n                +ADw-category a:name+AD0-'android.intent.category.LAUNCHER'/+AD4AXA-n            +ADw-/intent+AC0-filter+AD4AXA-n        +ADw-/activity+AC0-alias+AD4AXA-n", "Can't find launcher app through android.intent.category.HOME category and android.intent.action.MAIN action", '+ACI-More than one BroadcastReceiver that handles android.intent.action.MEDIA+AF8-BUTTON was found', "Please replace '+AFw-xe3+AFw-x80+AFw-x90+AFw-xe5+AFw-xba+AFw-x94+AFw-xe7+AFw-x94+AFw-xa8+AFw-xe5+AFw-x8c+AFw-x85+AFw-xe5+AFw-x90+AFw-x8d+AFw-xe3+AFw-x80+AFw-x91.intent.action.COCKROACH' with application's packageName for UmengService in AndroidManifest+ACE-", "Please replace '+AFw-xe3+AFw-x80+AFw-x90+AFw-xe5+AFw-xba+AFw-x94+AFw-xe7+AFw-x94+AFw-

,Unnamed: 0,' a:targetActivity+AD0-'com.mplus.lib.ui.main.MainActivity' tools:targetApi+AD0-'25'+AD4AXA-n +ADw-intent+AC0-filter+AD4AXA-n +ADw-action a:name+AD0-'android.intent.action.MAIN'/+AD4AXA-n +ADw-category a:name+AD0-'android.intent.category.LAUNCHER'/+AD4AXA-n +ADw-/intent+AC0-filter+AD4AXA-n +ADw-/activity+AC0-alias+AD4AXA-n,Can't find launcher app through android.intent.category.HOME category and android.intent.action.MAIN action,+ACI-More than one BroadcastReceiver that handles android.intent.action.MEDIA+AF8-BUTTON was found,Please replace '+AFw-xe3+AFw-x80+AFw-x90+AFw-xe5+AFw-xba+AFw-x94+AFw-xe7+AFw-x94+AFw-xa8+AFw-xe5+AFw-x8c+AFw-x85+AFw-xe5+AFw-x90+AFw-x8d+AFw-xe3+AFw-x80+AFw-x91.intent.action.COCKROACH' with application's packageName for UmengService in AndroidManifest+ACE-,Please replace '+AFw-xe3+AFw-x80+AFw-x90+AFw-xe5+AFw-xba+AFw-x94+AFw-xe7+AFw-x94+AFw-xa8+AFw-xe5+AFw-x8c+AFw-x85+AFw-xe5+AFw-x90+AFw-x8d+AFw-xe3+AFw-x80+AFw-x91.intent.action.COMMAND' with application's packageName for RegistrationReceiver in AndroidManifest+ACE-,Please replace '+AFw-xe3+AFw-x80+AFw-x90+AFw-xe5+AFw-xba+AFw-x94+AFw-xe7+AFw-x94+AFw-xa8+AFw-xe5+AFw-x8c+AFw-x85+AFw-xe5+AFw-x90+AFw-x8d+AFw-xe3+AFw-x80+AFw-x91.intent.action.START' with application's packageName for UmengService in AndroidManifest+ACE-,+ACI-To prevent external tampering to your app's notifications,+ACI-' +ADw-action android:name+AD0AIgAi-android.intent.action.ANY+AF8-DATA+AF8-STATE+ACIAIg- /+AD4AXA-n +ADw-/intent+AC0-filter+AD4AXA-n'+ACI-,+ACI-' +ADw-action android:name+AD0AIgAi-android.intent.action.BOOT+AF8-COMPLETED+ACIAIg- /+AD4AXA-n +ADw-action android:name+AD0AIgAi-android.net.conn.CONNECTIVITY+AF8-CHANGE+ACIAIg- /+AD4AXA-n'+ACI-,...,com.yahoo.mail.APPWIDGET_COMPOSE,com.yahoo.mail.data.PURGE_VACUUM,com.yahoo.mail.command.SEND_MESSAGE,com.yahoo.mail.command.MESSAGE_IN_OUTBOX_TOO_LONG,com.yahoo.doubleplay.notifications.ACTION_BREAKING_NEWS_NOTIFICATION_RECEIVED,com.yahoo.doubleplay.notifications.ACTION_TOP_NEWS_NOTIFICATION_RECEIVED,Lcom/medibang/android/paint/tablet/model/information/Information; => setIntenturl => iput-object => Lcom/medibang/android/paint/tablet/model/information/Information;->intenturl Ljava/lang/String;,Lcom/medibang/android/paint/tablet/model/information/Information; => getIntenturl => iget-object => Lcom/medibang/android/paint/tablet/model/information/Information;->intenturl Ljava/lang/String;,org.strongswan.android.action.START_PROFILE,com.karttuner.racemonitor.permission.C2D_MESSAGE
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Count rows without loading the dataset into RAM
with open(DATA_PATH, "rb") as f:
    row_count = sum(1 for _ in f) - 1

print("Approximate number of data rows:", row_count)

Approximate number of data rows: 11598


In [7]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Dataset/feature_vectors_static.csv"

# Read only the header — NO dataset rows
columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()

print("Total columns:", len(columns))

print("\nLast 30 columns:")
for i, col in enumerate(columns[-30:], start=len(columns)-30):
    print(i, repr(col))

Total columns: 50621

Last 30 columns:
50591 'com.google.android.alarmclock.ALARM_ALERT'
50592 'com.sec.android.app.clockpackage.ALARM_ALERT'
50593 'com.samsung.sec.android.clockpackage.alarm.ALARM_ALERT'
50594 'com.htc.android.worldclock.ALARM_ALERT'
50595 'com.htc.android.ALARM_ALERT'
50596 'com.sonyericsson.alarm.ALARM_ALERT'
50597 'com.sonyericsson.organizer.ALARM_ALERT'
50598 'zte.com.cn.alarmclock.ALARM_ALERT'
50599 'com.motorola.alarmclock.ALARM_ALERT'
50600 'com.motorola.blur.alarmclock.ALARM_ALERT'
50601 'com.lge.alarm.alarmclocknew.ALARM_ALERT'
50602 'com.mobitobi.android.gentlealarm.ALARM_INFO'
50603 'com.splunchy.android.alarmclock.ALARM_ALERT'
50604 'app.medicalid.CHANGE_LANG'
50605 'com.imo.android.imoimbeta.permission.C2D_MESSAGE'
50606 'com.bnpp.login.utils.LoginTimerService$IdentifiedSessionExpiredReceiver'
50607 'com.bnpp.login.authenticatedsession.EasyBankingAppSession$UITimeoutReceiver'
50608 'com.yahoo.mobile.client.android.mail.permission.C2D_MESSAGE'
50609 'com.y

In [8]:
print("\nPossible label columns:")

keywords = ["label", "class", "malware", "target", "category", "type", "benign"]

for col in columns:
    if any(k in col.lower() for k in keywords):
        print(repr(col))


Possible label columns:
"' a:targetActivity+AD0-'com.mplus.lib.ui.main.MainActivity' tools:targetApi+AD0-'25'+AD4AXA-n            +ADw-intent+AC0-filter+AD4AXA-n                +ADw-action a:name+AD0-'android.intent.action.MAIN'/+AD4AXA-n                +ADw-category a:name+AD0-'android.intent.category.LAUNCHER'/+AD4AXA-n            +ADw-/intent+AC0-filter+AD4AXA-n        +ADw-/activity+AC0-alias+AD4AXA-n"
"Can't find launcher app through android.intent.category.HOME category and android.intent.action.MAIN action"
"+ACI-'    +ADw-action android:name+AD0AIgAi-com.google.android.c2dm.intent.REGISTRATION+ACIAIg- /+AD4AXA-n    +ADw-category android:name+AD0AIgAi-'+ACI-"
"'+ACU-1+ACQ-s definition in AndroidManifest.xml missing intent filter for com.amazon.device.messaging.intent.RECEIVE with category +ACU-2+ACQ-s'"
"'+ACU-1+ACQ-s definition in AndroidManifest.xml missing intent filter for com.amazon.device.messaging.intent.REGISTRATION with category +ACU-2+ACQ-s'"
"'+ACU-1+ACQ-s definition

In [9]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Dataset/feature_vectors_static.csv"

# Read ONLY the first column / metadata candidates
df_meta = pd.read_csv(
    DATA_PATH,
    usecols=[0],
    low_memory=False
)

print("First column name:", df_meta.columns[0])
print("\nFirst 20 values:")
print(df_meta.head(20).to_string(index=False))

print("\nUnique values:")
print(df_meta.iloc[:, 0].value_counts(dropna=False).head(20))

First column name: Unnamed: 0

First 20 values:
 Unnamed: 0
          0
          1
          2
          3
          4
          5
          6
          7
          8
          9
         10
         11
         12
         13
         14
         15
         16
         17
         18
         19

Unique values:
Unnamed: 0
11597    1
0        1
1        1
2        1
3        1
4        1
5        1
6        1
7        1
8        1
9        1
10       1
11       1
12       1
11581    1
11580    1
11579    1
11578    1
11577    1
11576    1
Name: count, dtype: int64


In [10]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Dataset/feature_vectors_static.csv"

# Read only the first row to inspect metadata structure
sample = pd.read_csv(DATA_PATH, nrows=10)

print("Shape:", sample.shape)

print("\nFirst 10 column names:")
for i, c in enumerate(sample.columns[:10]):
    print(i, repr(c))

print("\nFirst column datatype:")
print(sample.iloc[:, 0].dtype)

print("\nFirst column sample:")
print(sample.iloc[:, 0].tolist())

Shape: (10, 50621)

First 10 column names:
0 'Unnamed: 0'
1 "' a:targetActivity+AD0-'com.mplus.lib.ui.main.MainActivity' tools:targetApi+AD0-'25'+AD4AXA-n            +ADw-intent+AC0-filter+AD4AXA-n                +ADw-action a:name+AD0-'android.intent.action.MAIN'/+AD4AXA-n                +ADw-category a:name+AD0-'android.intent.category.LAUNCHER'/+AD4AXA-n            +ADw-/intent+AC0-filter+AD4AXA-n        +ADw-/activity+AC0-alias+AD4AXA-n"
2 "Can't find launcher app through android.intent.category.HOME category and android.intent.action.MAIN action"
3 '+ACI-More than one BroadcastReceiver that handles android.intent.action.MEDIA+AF8-BUTTON was found'
4 "Please replace '+AFw-xe3+AFw-x80+AFw-x90+AFw-xe5+AFw-xba+AFw-x94+AFw-xe7+AFw-x94+AFw-xa8+AFw-xe5+AFw-x8c+AFw-x85+AFw-xe5+AFw-x90+AFw-x8d+AFw-xe3+AFw-x80+AFw-x91.intent.action.COCKROACH' with application's packageName for UmengService in AndroidManifest+ACE-"
5 "Please replace '+AFw-xe3+AFw-x80+AFw-x90+AFw-xe5+AFw-xba+AFw-x94+AFw-xe7+A

In [6]:
# 2. Target detection
TARGET_COLUMN = None  # Set explicitly if automatic detection is wrong.

candidates = ["Class","class","Label","label","Category","category","Malware","malware","Type","type"]
matches = [c for c in candidates if c in df.columns]
if TARGET_COLUMN is None:
    if not matches:
        raise ValueError("Set TARGET_COLUMN manually.")
    TARGET_COLUMN = matches[0]

print("Target:", TARGET_COLUMN)
display(df[TARGET_COLUMN].value_counts(dropna=False))

NameError: name 'df' is not defined

In [ ]:
# 3. Clean labels and duplicates
data = df.drop_duplicates().dropna(subset=[TARGET_COLUMN]).copy()

label_map = {
    "benign":"Benign", "adware":"Adware", "banking":"Banking",
    "sms":"SMS", "sms malware":"SMS", "riskware":"Riskware"
}
data[TARGET_COLUMN] = (
    data[TARGET_COLUMN].astype(str).str.strip().str.lower()
    .map(lambda x: label_map.get(x, x.title()))
)

print(data.shape)
display(data[TARGET_COLUMN].value_counts())

In [ ]:
# 4. Feature construction
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler, LabelEncoder

DROP = {TARGET_COLUMN,"sha256","sha1","md5","filename","file_name",
        "package_name","app_name","apk_name"}

feature_cols = [c for c in data.select_dtypes(include=[np.number]).columns if c not in DROP]
if len(feature_cols) < 2:
    raise ValueError("Too few numeric features; inspect dataset schema.")

X_raw = data[feature_cols].replace([np.inf,-np.inf],np.nan)
X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

le = LabelEncoder()
y = le.fit_transform(data[TARGET_COLUMN].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)

print("Features:",len(feature_cols),"Classes:",class_names)

In [ ]:
# 5. Leakage-safe split: 70/15/15
Xtr0, Xtmp0, ytr, ytmp = train_test_split(
    X_raw,y,test_size=.30,stratify=y,random_state=SEED
)
Xva0, Xte0, yva, yte = train_test_split(
    Xtmp0,ytmp,test_size=.50,stratify=ytmp,random_state=SEED
)

# Fit selector ONLY on training data.
selector = VarianceThreshold(0.0)
Xtr_sel = selector.fit_transform(Xtr0)
Xva_sel = selector.transform(Xva0)
Xte_sel = selector.transform(Xte0)

selected_features = [feature_cols[i] for i,v in enumerate(selector.get_support()) if v]

# Scaled branch for SVM/DL; tree branch remains unscaled.
scaler = StandardScaler()
Xtr = scaler.fit_transform(Xtr_sel)
Xva = scaler.transform(Xva_sel)
Xte = scaler.transform(Xte_sel)

print(Xtr.shape, Xva.shape, Xte.shape)

In [ ]:
# 6. Class weights
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight(
    class_weight="balanced", classes=np.arange(n_classes), y=ytr
)
class_weights = {i:float(w) for i,w in enumerate(weights)}
sample_weights = np.array([class_weights[int(v)] for v in ytr])
print(dict(zip(class_names,weights)))

## Advanced classical ML
Tree models use the unscaled feature branch. Scaling is retained for SVM and neural networks.

In [ ]:
# 7. Advanced ML benchmark
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    HistGradientBoostingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, matthews_corrcoef, cohen_kappa_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

models = {
    "Random Forest": RandomForestClassifier(n_estimators=400,class_weight="balanced",n_jobs=-1,random_state=SEED),
    "Extra Trees": ExtraTreesClassifier(n_estimators=400,class_weight="balanced",n_jobs=-1,random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=500,max_depth=8,learning_rate=.05,
        subsample=.85,colsample_bytree=.85,objective="multi:softprob",
        num_class=n_classes,eval_metric="mlogloss",tree_method="hist",random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=500,learning_rate=.05,num_leaves=63,
        class_weight="balanced",random_state=SEED,verbosity=-1),
    "CatBoost": CatBoostClassifier(iterations=500,depth=8,learning_rate=.05,
        loss_function="MultiClass",verbose=False,random_seed=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,learning_rate=.08,max_leaf_nodes=63,random_state=SEED)
}

fitted = {}
rows = []
for name,m in models.items():
    if name in {"XGBoost","LightGBM","CatBoost","HistGradientBoosting"}:
        m.fit(Xtr_sel,ytr,sample_weight=sample_weights)
    else:
        m.fit(Xtr_sel,ytr)
    p = m.predict(Xte_sel); pr = m.predict_proba(Xte_sel)
    rows.append({
        "Model":name,"Accuracy":accuracy_score(yte,p),
        "Macro Precision":precision_score(yte,p,average="macro",zero_division=0),
        "Macro Recall":recall_score(yte,p,average="macro",zero_division=0),
        "Macro F1":f1_score(yte,p,average="macro",zero_division=0),
        "Weighted F1":f1_score(yte,p,average="weighted",zero_division=0),
        "MCC":matthews_corrcoef(yte,p),"Kappa":cohen_kappa_score(yte,p)
    })
    fitted[name]=m

ml_results = pd.DataFrame(rows).sort_values("Macro F1",ascending=False)
display(ml_results)
best_ml_name = ml_results.iloc[0].Model
best_ml = fitted[best_ml_name]
best_ml_proba = best_ml.predict_proba(Xte_sel)
print("Best ML:",best_ml_name)

In [ ]:
# 8. Stacking ensemble
stack = StackingClassifier(
    estimators=[
        ("xgb",XGBClassifier(n_estimators=350,max_depth=7,learning_rate=.05,
            subsample=.85,colsample_bytree=.85,objective="multi:softprob",
            num_class=n_classes,eval_metric="mlogloss",tree_method="hist",random_state=SEED)),
        ("lgbm",LGBMClassifier(n_estimators=350,learning_rate=.05,num_leaves=63,
            class_weight="balanced",random_state=SEED,verbosity=-1)),
        ("extra",ExtraTreesClassifier(n_estimators=300,class_weight="balanced",
            n_jobs=-1,random_state=SEED))
    ],
    final_estimator=LogisticRegression(max_iter=2000),
    stack_method="predict_proba",n_jobs=-1
)
stack.fit(Xtr_sel,ytr)
stack_proba = stack.predict_proba(Xte_sel)
print(classification_report(yte,stack.predict(Xte_sel),target_names=class_names,digits=4,zero_division=0))

## Deep learning
Four architectures are included: deep MLP, 1D CNN, feature-attention network, and autoencoder + classifier.

In [ ]:
# 9. PyTorch
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

Xtr_t=torch.tensor(Xtr,dtype=torch.float32); Xva_t=torch.tensor(Xva,dtype=torch.float32); Xte_t=torch.tensor(Xte,dtype=torch.float32)
ytr_t=torch.tensor(ytr,dtype=torch.long); yva_t=torch.tensor(yva,dtype=torch.long); yte_t=torch.tensor(yte,dtype=torch.long)

train_loader=DataLoader(TensorDataset(Xtr_t,ytr_t),batch_size=256,shuffle=True)
val_loader=DataLoader(TensorDataset(Xva_t,yva_t),batch_size=256)
INPUT_DIM=Xtr.shape[1]
print("Device:",DEVICE,"Input:",INPUT_DIM)

In [ ]:
# 10. Models
class DeepMLP(nn.Module):
    def __init__(self,d,k):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(d,512),nn.BatchNorm1d(512),nn.GELU(),nn.Dropout(.30),
            nn.Linear(512,256),nn.BatchNorm1d(256),nn.GELU(),nn.Dropout(.25),
            nn.Linear(256,128),nn.BatchNorm1d(128),nn.GELU(),nn.Dropout(.20),
            nn.Linear(128,64),nn.GELU(),nn.Linear(64,k))
    def forward(self,x): return self.net(x)

class CNN1D(nn.Module):
    def __init__(self,d,k):
        super().__init__()
        self.f=nn.Sequential(
            nn.Conv1d(1,64,5,padding=2),nn.BatchNorm1d(64),nn.GELU(),
            nn.Conv1d(64,128,5,padding=2),nn.BatchNorm1d(128),nn.GELU(),nn.Dropout(.2),
            nn.Conv1d(128,128,3,padding=1),nn.BatchNorm1d(128),nn.GELU(),
            nn.AdaptiveAvgPool1d(1))
        self.h=nn.Sequential(nn.Flatten(),nn.Linear(128,64),nn.GELU(),nn.Dropout(.2),nn.Linear(64,k))
    def forward(self,x): return self.h(self.f(x.unsqueeze(1)))

class FeatureAttention(nn.Module):
    def __init__(self,d,k,e=64,h=4):
        super().__init__()
        self.proj=nn.Linear(1,e)
        self.attn=nn.MultiheadAttention(e,h,batch_first=True,dropout=.1)
        self.n1=nn.LayerNorm(e); self.ff=nn.Sequential(nn.Linear(e,2*e),nn.GELU(),nn.Dropout(.15),nn.Linear(2*e,e))
        self.n2=nn.LayerNorm(e); self.head=nn.Sequential(nn.Linear(e,128),nn.GELU(),nn.Dropout(.2),nn.Linear(128,k))
    def forward(self,x):
        x=self.proj(x.unsqueeze(-1)); a,_=self.attn(x,x,x); x=self.n1(x+a); x=self.n2(x+self.ff(x))
        return self.head(x.mean(1))

class AEClassifier(nn.Module):
    def __init__(self,d,k,z=64):
        super().__init__()
        self.enc=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Dropout(.1),nn.Linear(256,128),nn.ReLU(),nn.Linear(128,z))
        self.dec=nn.Sequential(nn.Linear(z,128),nn.ReLU(),nn.Linear(128,256),nn.ReLU(),nn.Linear(256,d))
        self.cls=nn.Sequential(nn.Linear(z,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,k))
    def forward(self,x):
        z=self.enc(x); return self.cls(z),self.dec(z),z

mlp=DeepMLP(INPUT_DIM,n_classes).to(DEVICE)
cnn=CNN1D(INPUT_DIM,n_classes).to(DEVICE)
att=FeatureAttention(INPUT_DIM,n_classes).to(DEVICE)
ae=AEClassifier(INPUT_DIM,n_classes).to(DEVICE)
print("Models created.")

In [ ]:
# 11. Training utility
def train_classifier(model,epochs=40,lr=1e-3,patience=7):
    loss_fn=nn.CrossEntropyLoss(weight=torch.tensor([class_weights[i] for i in range(n_classes)],dtype=torch.float32,device=DEVICE))
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=.5,patience=2)
    best=-1; state=None; wait=0
    for ep in range(1,epochs+1):
        model.train()
        for xb,yb in train_loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            loss=loss_fn(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),2.0); opt.step()
        model.eval(); yt=[]; yp=[]
        with torch.no_grad():
            for xb,yb in val_loader:
                q=model(xb.to(DEVICE)).argmax(1).cpu().numpy()
                yp.extend(q); yt.extend(yb.numpy())
        f=f1_score(yt,yp,average="macro",zero_division=0); sched.step(f)
        if f>best:
            best=f; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else: wait+=1
        if ep==1 or ep%5==0: print(f"Epoch {ep:02d} | val macro-F1={f:.4f}")
        if wait>=patience: break
    model.load_state_dict(state)
    return model

mlp=train_classifier(mlp)
cnn=train_classifier(cnn,lr=8e-4)
att=train_classifier(att,lr=8e-4)

In [ ]:
# 12. Autoencoder + classifier training
def train_ae(model,epochs=40,lr=1e-3,alpha=.20,patience=7):
    ce=nn.CrossEntropyLoss(weight=torch.tensor([class_weights[i] for i in range(n_classes)],dtype=torch.float32,device=DEVICE))
    mse=nn.MSELoss(); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    best=-1; state=None; wait=0
    for ep in range(1,epochs+1):
        model.train()
        for xb,yb in train_loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
            logits,recon,_=model(xb); loss=ce(logits,yb)+alpha*mse(recon,xb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),2.0); opt.step()
        model.eval(); yt=[]; yp=[]
        with torch.no_grad():
            for xb,yb in val_loader:
                logits,_,_=model(xb.to(DEVICE)); yt.extend(yb.numpy()); yp.extend(logits.argmax(1).cpu().numpy())
        f=f1_score(yt,yp,average="macro",zero_division=0)
        if ep==1 or ep%5==0: print(f"Epoch {ep:02d} | val macro-F1={f:.4f}")
        if f>best:
            best=f; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else: wait+=1
        if wait>=patience: break
    model.load_state_dict(state); return model

ae=train_ae(ae)

In [ ]:
# 13. Prediction helpers
def torch_proba(model,X):
    model.eval(); out=[]
    loader=DataLoader(torch.tensor(X,dtype=torch.float32),batch_size=512)
    with torch.no_grad():
        for xb in loader:
            out.append(torch.softmax(model(xb.to(DEVICE)),1).cpu().numpy())
    return np.vstack(out)

mlp_proba=torch_proba(mlp,Xte)
cnn_proba=torch_proba(cnn,Xte)
att_proba=torch_proba(att,Xte)

def ae_predict(model,X):
    model.eval(); ps=[]; errs=[]
    loader=DataLoader(torch.tensor(X,dtype=torch.float32),batch_size=512)
    with torch.no_grad():
        for xb in loader:
            logits,recon,_=model(xb.to(DEVICE))
            ps.append(torch.softmax(logits,1).cpu().numpy())
            errs.append(((recon-xb.to(DEVICE))**2).mean(1).cpu().numpy())
    return np.vstack(ps),np.concatenate(errs)

ae_proba,recon_err=ae_predict(ae,Xte)

for name,pr in {"MLP":mlp_proba,"CNN":cnn_proba,"Attention":att_proba,"Autoencoder":ae_proba}.items():
    print("\n",name)
    print(classification_report(yte,pr.argmax(1),target_names=class_names,digits=4,zero_division=0))

## Hybrid ML + DL ensemble and uncertainty

In [ ]:
# 14. Probability ensemble
components={"Best ML":best_ml_proba,"Stacking":stack_proba,"MLP":mlp_proba,"CNN":cnn_proba,"Attention":att_proba,"Autoencoder":ae_proba}
weights={"Best ML":.20,"Stacking":.20,"MLP":.15,"CNN":.15,"Attention":.15,"Autoencoder":.15}
hybrid=sum(weights[k]*v for k,v in components.items())
hybrid/=hybrid.sum(1,keepdims=True)
hybrid_pred=hybrid.argmax(1)

print(classification_report(yte,hybrid_pred,target_names=class_names,digits=4,zero_division=0))
print("Hybrid Macro-F1:",f1_score(yte,hybrid_pred,average="macro"))

In [ ]:
# 15. Uncertainty
def entropy(p):
    p=np.clip(p,1e-12,1); return -(p*np.log(p)).sum(1)

uncertainty=entropy(hybrid)/np.log(n_classes)
confidence=hybrid.max(1)

print("Mean normalized uncertainty:",uncertainty.mean())

## SHAP explainability

In [ ]:
# 16. SHAP on the best tree model
import shap
try:
    explainer=shap.TreeExplainer(best_ml)
    n=min(1000,len(Xte_sel))
    sv=explainer.shap_values(Xte_sel[:n])
    if isinstance(sv,list):
        abs_sv=np.mean([np.abs(v) for v in sv],axis=0)
    else:
        a=np.asarray(sv); abs_sv=np.mean(np.abs(a),axis=2) if a.ndim==3 else np.abs(a)
    shap_importance=pd.DataFrame({
        "feature":selected_features,
        "mean_abs_shap":abs_sv.mean(0)
    }).sort_values("mean_abs_shap",ascending=False)
    display(shap_importance.head(25))
except Exception as e:
    print("SHAP unavailable for this model/environment:",e)

## Security risk score

This is an operational prioritization score. It combines malware probability, class severity, anomaly/reconstruction signal, and uncertainty.

In [ ]:
# 17. Engineered 0-100 risk score
severity={}
for c in class_names:
    x=c.lower()
    severity[c] = 0.0 if x=="benign" else 0.45 if x=="adware" else 0.65 if x=="sms" else 0.75 if x=="riskware" else 0.95 if x=="banking" else 0.70
sev=np.array([severity[c] for c in class_names])

malware_prob = 1-hybrid[:,class_names.index("Benign")] if "Benign" in class_names else confidence
rmin,rmax=np.percentile(recon_err,1),np.percentile(recon_err,99)
recon_risk=np.clip((recon_err-rmin)/max(rmax-rmin,1e-12),0,1)
class_risk=hybrid@sev

risk_score=np.clip(100*(.50*malware_prob+.25*class_risk+.15*recon_risk+.10*uncertainty),0,100)

def risk_level(s):
    return "LOW" if s<=20 else "MODERATE" if s<=40 else "MEDIUM" if s<=60 else "HIGH" if s<=80 else "CRITICAL"

risk_df=pd.DataFrame({
    "Predicted_Class":[class_names[i] for i in hybrid_pred],
    "Confidence":confidence,
    "Malware_Probability":malware_prob,
    "Reconstruction_Risk":recon_risk,
    "Uncertainty":uncertainty,
    "Risk_Score":risk_score
})
risk_df["Risk_Level"]=risk_df.Risk_Score.apply(risk_level)
display(risk_df.head(20))

## Advanced regression models

The regression stage predicts the continuous operational risk score. For publication, use independently annotated expert risk labels rather than a score derived from the same classifier outputs.

In [ ]:
# 18. Regression benchmark
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

R=np.column_stack([hybrid,malware_prob,recon_risk,uncertainty])
Rtr,Rte,rtr,rte=train_test_split(R,risk_score,test_size=.20,random_state=SEED)

regs={
"Random Forest Regressor":RandomForestRegressor(n_estimators=400,n_jobs=-1,random_state=SEED),
"Extra Trees Regressor":ExtraTreesRegressor(n_estimators=400,n_jobs=-1,random_state=SEED),
"XGBoost Regressor":XGBRegressor(n_estimators=500,max_depth=7,learning_rate=.05,subsample=.85,
    colsample_bytree=.85,objective="reg:squarederror",tree_method="hist",random_state=SEED),
"LightGBM Regressor":LGBMRegressor(n_estimators=500,learning_rate=.05,num_leaves=63,random_state=SEED,verbosity=-1),
"CatBoost Regressor":CatBoostRegressor(iterations=500,depth=8,learning_rate=.05,loss_function="RMSE",verbose=False,random_seed=SEED),
"HistGradientBoosting Regressor":HistGradientBoostingRegressor(max_iter=300,learning_rate=.08,max_leaf_nodes=63,random_state=SEED)
}
reg_rows=[]; reg_fitted={}
for name,m in regs.items():
    m.fit(Rtr,rtr); p=np.clip(m.predict(Rte),0,100)
    reg_rows.append({"Model":name,"MAE":mean_absolute_error(rte,p),"RMSE":mean_squared_error(rte,p)**.5,"R2":r2_score(rte,p)})
    reg_fitted[name]=m
reg_results=pd.DataFrame(reg_rows).sort_values("RMSE")
display(reg_results)
best_reg=reg_fitted[reg_results.iloc[0].Model]

## Final comparison and artifacts

In [ ]:
# 19. Final classification table
final_rows=ml_results.copy()
final_rows=pd.concat([final_rows,pd.DataFrame([{
    "Model":"Hybrid ML + DL Ensemble",
    "Accuracy":accuracy_score(yte,hybrid_pred),
    "Macro Precision":precision_score(yte,hybrid_pred,average="macro",zero_division=0),
    "Macro Recall":recall_score(yte,hybrid_pred,average="macro",zero_division=0),
    "Macro F1":f1_score(yte,hybrid_pred,average="macro",zero_division=0),
    "Weighted F1":f1_score(yte,hybrid_pred,average="weighted",zero_division=0),
    "MCC":matthews_corrcoef(yte,hybrid_pred),
    "Kappa":cohen_kappa_score(yte,hybrid_pred)
}])],ignore_index=True).sort_values("Macro F1",ascending=False)
display(final_rows)

# Confusion matrix
cm=confusion_matrix(yte,hybrid_pred)
print("Confusion matrix:\n",cm)

In [ ]:
# 20. Save artifacts for the Agentic-AI layer
joblib.dump({
    "selector":selector,"scaler":scaler,"label_encoder":le,
    "feature_columns":feature_cols,"selected_features":selected_features,
    "class_names":class_names
},ARTIFACT_DIR/"preprocessing_bundle.joblib")
joblib.dump(fitted,ARTIFACT_DIR/"advanced_ml_models.joblib")
joblib.dump(stack,ARTIFACT_DIR/"stacking_model.joblib")
joblib.dump(reg_fitted,ARTIFACT_DIR/"risk_regressors.joblib")

torch.save(mlp.state_dict(),ARTIFACT_DIR/"deep_mlp.pt")
torch.save(cnn.state_dict(),ARTIFACT_DIR/"cnn1d.pt")
torch.save(att.state_dict(),ARTIFACT_DIR/"attention_model.pt")
torch.save(ae.state_dict(),ARTIFACT_DIR/"autoencoder_classifier.pt")

metadata={
    "seed":SEED,"classes":class_names,"input_dim":int(INPUT_DIM),
    "selected_feature_count":len(selected_features),
    "best_ml_model":best_ml_name,
    "best_risk_regressor":reg_results.iloc[0].Model,
    "ensemble_weights":weights,"severity_map":severity
}
with open(ARTIFACT_DIR/"model_metadata.json","w") as f: json.dump(metadata,f,indent=2)

final_rows.to_csv(ARTIFACT_DIR/"classification_results.csv",index=False)
reg_results.to_csv(ARTIFACT_DIR/"regression_results.csv",index=False)
print("Saved:",ARTIFACT_DIR.resolve())

## Agentic-AI handoff

The next module should load these artifacts and expose the prediction/risk evidence as structured tool output to LangChain agents. GPT should interpret evidence and generate an analyst report; it should not replace the malware classifier.

In [ ]:
# 21. Structured payload helper
def build_agent_payload_from_processed(raw_feature_vector, app_identifier="unknown_app"):
    raw = np.asarray(raw_feature_vector).reshape(1,-1)
    selected = selector.transform(pd.DataFrame(raw,columns=feature_cols))
    scaled = scaler.transform(selected)

    probs=[
        best_ml.predict_proba(selected)[0],
        stack.predict_proba(selected)[0],
        torch_proba(mlp,scaled)[0],
        torch_proba(cnn,scaled)[0],
        torch_proba(att,scaled)[0]
    ]
    ae_p,ae_e=ae_predict(ae,scaled)
    probs.append(ae_p[0])

    combined=sum(weights[k]*p for k,p in zip(weights.keys(),probs))
    combined/=combined.sum()
    idx=int(combined.argmax())
    mal=float(1-combined[class_names.index("Benign")]) if "Benign" in class_names else float(combined.max())
    u=float(entropy(combined.reshape(1,-1))[0]/np.log(n_classes))
    rr=float(np.clip((ae_e[0]-rmin)/max(rmax-rmin,1e-12),0,1))
    cr=float(combined@sev)
    rs=float(np.clip(100*(.50*mal+.25*cr+.15*rr+.10*u),0,100))

    return {
        "application":app_identifier,
        "prediction":class_names[idx],
        "confidence":float(combined.max()),
        "class_probabilities":{class_names[i]:float(combined[i]) for i in range(n_classes)},
        "malware_probability":mal,
        "reconstruction_risk":rr,
        "prediction_uncertainty":u,
        "risk_score":rs,
        "risk_level":risk_level(rs),
        "recommended_next_step":"QUARANTINE_AND_MANUAL_ANALYSIS" if rs>=80 else "ENHANCED_ANALYSIS" if rs>=60 else "STANDARD_ANALYSIS"
    }

print("LangChain-ready structured inference interface created.")

### Recommended next stage

- Add Optuna hyperparameter optimization.
- Add repeated stratified cross-validation.
- Calibrate class probabilities.
- Replace engineered risk labels with independent analyst annotations.
- Connect the structured inference payload to LangChain agents.
- Add controlled Prowler / Scout Suite / AWS Security Hub tools.
- Generate evidence-backed analyst reports with remediation.